# 02. No-Show Classification Model Development
## Binary Classification & Risk Stratification

### Objectives:
1. Build an end-to-end preprocessing and feature engineering pipeline without data leakage.
2. Implement and benchmark at least three classification algorithms:
   - **Logistic Regression** (Linear baseline with balanced class weights)
   - **Random Forest Classifier** (Ensemble tree-based model)
   - **XGBoost Classifier** (Gradient boosted trees with scale_pos_weight)
3. Optimize classification thresholds on training folds to maximize F1-score and discrimination.
4. Evaluate models against targets: **F1-Score > 0.70** and **ROC-AUC > 0.75**.
5. Extract global feature importances and export the trained artifact to `models/no_show_model.joblib`.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

from src.data import prepare_classification_data
from src.classification import train_and_compare_models, extract_feature_importance, save_classifier_artifact

sns.set_theme(style='whitegrid')
%matplotlib inline


### 1. Data Preparation & Stratified Split

In [ ]:
X_train, X_test, y_train, y_test = prepare_classification_data(
    filepath='../data/raw.csv',
    test_size=0.20,
    random_state=42
)

print(f"X_train shape: {X_train.shape}, y_train mean: {y_train.mean():.4f}")
print(f"X_test shape:  {X_test.shape}, y_test mean:  {y_test.mean():.4f}")


### 2. Model Training & Comparison

In [ ]:
results, best_model_name, best_pipeline = train_and_compare_models(
    X_train, y_train, X_test, y_test, output_dir='../models'
)

# Tabulate results
results_df = pd.DataFrame(results).T[['roc_auc', 'f1_score_tuned', 'optimal_threshold', 'f1_score_default', 'precision_tuned', 'recall_tuned', 'accuracy_tuned']]
results_df


### 3. Confusion Matrix and Error Analysis

In [ ]:
best_meta = results[best_model_name]
cm = np.array(best_meta['confusion_matrix_tuned'])

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Show', 'No-Show'], yticklabels=['Show', 'No-Show'])
plt.title(f'Confusion Matrix: {best_model_name} (Threshold: {best_meta["optimal_threshold"]:.2f})')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()


### 4. Feature Importance & Interpretation

In [ ]:
feat_imp = extract_feature_importance(best_pipeline)

plt.figure(figsize=(9, 6))
top15 = feat_imp.head(15).iloc[::-1]
plt.barh(top15['feature'], top15['importance'], color='#2b5c8f')
plt.title(f'Top 15 Feature Importances ({best_model_name})')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()


### 5. Save Model Artifact

In [ ]:
save_classifier_artifact(
    pipeline=best_pipeline,
    metadata={
        'model_name': best_model_name,
        'optimal_threshold': best_meta['optimal_threshold'],
        'metrics': best_meta,
        'all_models': results
    },
    filepath='../models/no_show_model.joblib'
)
print("Pipeline artifact saved successfully!")
